<a href="https://colab.research.google.com/github/ajaykuk/JavaPrograms/blob/master/Agentic_AI_Day1_Day2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Agentic AI — Day 1 & 2 Interactive Notebook
### Theory · Diagrams · Graded Quizzes · Hands-on Activities

| | |
|---|---|
| **Duration** | 120 minutes |
| **Tools** | n8n + OpenAI / Gemini |
| **Prerequisites** | No prior ML required |
| **Quizzes** | 4 graded quizzes, 21 points total |

---

## 📋 Contents
1. [What is an AI Agent? — The OTA loop](#1)
2. [LLM Vocabulary — 8 key terms](#2)
3. [How LLMs Work — Transformer walkthrough](#3)
4. [Quiz 1 — Term Matching Sprint](#4)
5. [Context Stuffing vs RAG — The critical distinction](#5)
6. [Quiz 2 — Context Window & RAG](#6)
7. [Building your first n8n agent — Walkthrough](#7)
8. [Quiz 3 — Agent Architecture](#8)
9. [Why Build vs. Use ChatGPT?](#9)
10. [Quiz 4 — Harder MCQs (graded)](#10)
11. [Homework & Extension Challenges](#11)
12. [Final Score Dashboard](#12)

> **How to use this notebook:** Run each cell top-to-bottom with `Shift+Enter`. Quiz cells are interactive — follow the prompts. You don't need any API keys to complete the theory and quizzes.

In [ ]:
# ─────────────────────────────────────────────
# SETUP — Run this cell first (Shift+Enter)
# ─────────────────────────────────────────────
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
import json, time

# Shared score tracker
SCORES = {"q1_match": 0, "q2": 0, "q3": 0, "q4": 0}
SCORE_MAX = {"q1_match": 8, "q2": 4, "q3": 4, "q4": 5}

def color_bar(pct, width=40):
    filled = int(pct / 100 * width)
    bar = '█' * filled + '░' * (width - filled)
    return f"|{bar}| {pct:.0f}%"

def grade_label(pct):
    if pct >= 85: return "🏆 Outstanding"
    if pct >= 70: return "✅ Solid"
    if pct >= 50: return "📈 Good progress"
    return "📚 Keep studying"

print("✅ Setup complete! Proceed to Cell 1.")

✅ Setup complete! Proceed to Cell 1.


<a id='1'></a>
---
## 🔁 1. What is an AI Agent? — The Observe → Think → Act Loop

An **AI agent** is any system that *perceives* its environment, *reasons* about what to do, and *takes an action* — then loops.

```
 ┌──────────────┐      ┌──────────────┐      ┌──────────────┐
 │   OBSERVE    │ ───▶ │    THINK     │ ───▶ │     ACT      │
 │              │      │              │      │              │
 │ Read email,  │      │  LLM reasons │      │ Send email,  │
 │ PDF, sensor  │      │  and plans   │      │ call API     │
 └──────────────┘      └──────────────┘      └──────────────┘
         ▲                                          │
         └──────────── loop repeats ◀───────────────┘
```

### The key insight
The **LLM is the Think step**. It doesn't automatically observe or act. You connect it to:
- **Inputs** (documents, emails, webhooks) → the Observe step
- **Outputs** (email nodes, APIs, databases) → the Act step

Using a tool like **n8n**, you wire these three steps together visually.

> 💡 **Real example from today's class:**  
> `Manual Trigger` (Observe) → `OpenAI node reads document` (Think) → `Gmail node sends answer` (Act)  
> That's a complete agent in just three nodes.

### Desktop AI vs. Agents you build

| | Desktop AI (ChatGPT / Claude.ai) | Agent you build (n8n + API) |
|---|---|---|
| **Stops at** | Think — gives you text | Act — does something with the answer |
| **Integrations** | None | Any database, CRM, email, Slack |
| **Trigger** | You type | Events, schedules, HTTP calls |
| **Cost** | ~$20/month subscription | ~$0.01/run |
| **IP control** | None | Fully yours |

In [ ]:
# ─────────────────────────────────────────────
# CELL 1 — Map YOUR use case to the OTA loop
# ─────────────────────────────────────────────
print("🗺️  Map your Think Day use case to the Observe → Think → Act loop")
print("=" * 60)

observe = input("\n👁️  OBSERVE — What does your agent read or receive?\n   (e.g. a PDF report, an incoming email, a form submission)\n   > ")
think   = input("\n🧠 THINK   — What question should the LLM answer?\n   (e.g. 'What is the main risk?', 'Summarise in 3 bullets')\n   > ")
act     = input("\n⚡ ACT     — What should the agent do with the answer?\n   (e.g. send an email, post to Slack, update a spreadsheet)\n   > ")

print("\n" + "=" * 60)
print("✅ Your agent blueprint:")
print(f"""
 ┌────────────────────────────┐
 │  OBSERVE: {observe[:28]:<28}│
 └────────────┬───────────────┘
              │
 ┌────────────▼───────────────┐
 │  THINK:   {think[:28]:<28}│
 └────────────┬───────────────┘
              │
 ┌────────────▼───────────────┐
 │  ACT:     {act[:28]:<28}│
 └────────────────────────────┘
""")
print("Save this — you'll prototype it in Week 11!")

🗺️  Map your Think Day use case to the Observe → Think → Act loop


<a id='2'></a>
---
## 📚 2. LLM Vocabulary — The 8 Terms Every Agent Builder Needs

Study these carefully — Quiz 1 tests all of them.

---

### 🔧 TRANSFORMER
The neural network architecture underlying every major LLM. Introduced in Google's 2017 paper *"Attention Is All You Need"*. Every model you'll use — GPT-4, Claude, Gemini — is a transformer. Not the movie robots.

---

### 👁️ ATTENTION
The mechanism that lets the model weigh how relevant every word is to every other word — **simultaneously**, not one-by-one. This is why LLMs handle long documents better than older RNNs.

*Example: In "The cat sat on the mat", when processing "sat", attention scores are highest for "cat" (who sat?) and "mat" (sat where?).*

---

### 🌐 PRE-TRAINING
Phase 1: train on billions of tokens from the internet and books. Billions of parameters adjusted. **Costs millions of dollars. Done once by the lab** (OpenAI, Anthropic, Google). You benefit from it every time you call the API.

---

### 🎯 RLHF (Reinforcement Learning from Human Feedback)
Phase 2 — post-training. Human raters score model outputs; a reward model is trained; the LLM is fine-tuned to score higher. **This is why Claude sounds polite and ChatGPT follows instructions.** Without RLHF, a pre-trained model just predicts text — it doesn't answer questions helpfully.

---

### ⚙️ GPU (Graphics Processing Unit)
Chips originally designed for rendering 3D graphics, repurposed for matrix math. **A 70B-parameter model needs ~140 GB of GPU memory to run.** This is why cloud APIs exist — you rent the GPU for milliseconds at a time.

---

### 💬 PROMPT
Everything you send to the model: system instructions, examples, context, and the user's question. **Prompting IS programming for agents.** A well-crafted prompt can transform the same model from useless to exceptional.

---

### 🪟 CONTEXT WINDOW
The model's working memory — all tokens it can see at once. **Claude 3.5 Sonnet: 200K tokens (~150K words).** Everything must fit here: instructions, conversation history, documents, and the output. Nothing persists between API calls.

---

### 🔍 RAG (Retrieval-Augmented Generation)
Instead of stuffing the whole document into the prompt, **chunk it → embed it → store in a vector database → retrieve only the relevant chunks at query time.** The LLM only sees the relevant 2–3 pages, not the entire 500-page document.

---

> 🍳 **Memory tip:** Think of it like cooking.  
> **Pre-training** = teaching a chef every cuisine (years, expensive).  
> **RLHF** = restaurant etiquette training (weeks).  
> **Prompt** = today's order.  
> **Context window** = the chef's working memory (forgets between shifts).

<a id='3'></a>
---
## 🧠 3. How LLMs Work — Transformer Walkthrough

When you send text to an LLM, it goes through 5 stages:

```
INPUT TEXT
    │
    ▼
┌─────────────────────────────────────────────────┐
│  STAGE 1: TOKENISATION                          │
│  "The cat sat" → [464, 3797, 3332]              │
│  ~1 token ≈ ¾ of a word on average              │
└──────────────────────┬──────────────────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────────┐
│  STAGE 2: EMBEDDINGS                            │
│  Each token ID → dense vector (e.g. 4096 dims)  │
│  "cat" → [0.23, -0.71, 0.04, 0.88, ...]         │
│  Similar words have similar vectors              │
└──────────────────────┬──────────────────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────────┐
│  STAGE 3: ATTENTION LAYERS                      │
│  Each token asks: which other tokens matter      │
│  most for understanding ME?                      │
│                                                  │
│  "sat" → cat [0.72] mat [0.68] the [0.12]       │
│           ^^^^^ subject      ^^^^^ location      │
└──────────────────────┬──────────────────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────────┐
│  STAGE 4: STACKED LAYERS (32–96 deep)           │
│  Layer 1  → basic word features                 │
│  Layer 8  → syntax and grammar                  │
│  Layer 32 → abstract reasoning, world knowledge │
└──────────────────────┬──────────────────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────────┐
│  STAGE 5: NEXT-TOKEN PREDICTION                 │
│  Output: probability distribution over ~50K     │
│  tokens. Sample from top candidates.            │
│                                                  │
│  "The cat sat on the ___"                        │
│   mat   72% ██████████████                       │
│   floor 12% ███                                  │
│   chair  6% █                                    │
└─────────────────────────────────────────────────┘
```

### Why does temperature matter?
- **Temperature = 0**: Always picks the highest-probability token (deterministic). Use for fact extraction.
- **Temperature = 1**: Samples proportionally — creative but variable. Use for creative writing.
- **Temperature > 1**: Very random. Usually not useful.

### Key insight: LLMs have no memory between calls
Each API call is completely **stateless**. The model processes all tokens in a single forward pass, outputs the response, and forgets everything. If you want memory between calls, *you* have to store it and pass it back in the next prompt.

In [ ]:
# ─────────────────────────────────────────────
# CELL 3 — Token & context window calculator
# ─────────────────────────────────────────────
print("🪟 Context Window & Token Calculator")
print("=" * 50)

models = {
    "GPT-4o":          {"tokens": 128_000, "cost_per_1k_in": 0.0025, "cost_per_1k_out": 0.010},
    "GPT-4o mini":     {"tokens":  128_000, "cost_per_1k_in": 0.00015,"cost_per_1k_out": 0.0006},
    "Claude 3.5 Sonnet":{"tokens": 200_000, "cost_per_1k_in": 0.003,  "cost_per_1k_out": 0.015},
    "Gemini 1.5 Pro":  {"tokens": 1_000_000,"cost_per_1k_in": 0.00125,"cost_per_1k_out": 0.005},
    "Gemini 1.5 Flash":{"tokens": 1_000_000,"cost_per_1k_in": 0.000075,"cost_per_1k_out":0.0003},
}

print("\n📊 Model comparison:")
print(f"{'Model':<22} {'Context window':>16} {'≈ Words':>10} {'$/1K in':>9} {'$/1K out':>10}")
print("-" * 72)
for name, m in models.items():
    words = int(m['tokens'] * 0.75)
    print(f"{name:<22} {m['tokens']:>14,} tk {words:>9,} wd  ${m['cost_per_1k_in']:>7.5f}  ${m['cost_per_1k_out']:>7.4f}")

print("\n" + "─" * 50)
print("💰 Cost estimator for your agent:")
try:
    doc_words  = int(input("\nHow many words is your document? (e.g. 5000): ") or 5000)
    runs_month = int(input("How many times will you run the agent per month? (e.g. 100): ") or 100)
    model_name = input("Which model? (GPT-4o / GPT-4o mini / Claude 3.5 Sonnet / Gemini 1.5 Flash) [GPT-4o mini]: ").strip() or "GPT-4o mini"

    m = models.get(model_name, models["GPT-4o mini"])
    doc_tokens   = int(doc_words / 0.75)
    prompt_tokens = doc_tokens + 300  # system prompt + question overhead
    output_tokens = 200               # typical answer
    cost_per_run  = (prompt_tokens / 1000 * m["cost_per_1k_in"]) + (output_tokens / 1000 * m["cost_per_1k_out"])
    cost_month    = cost_per_run * runs_month

    fits = doc_tokens <= m["tokens"]
    print(f"""
┌────────────────────────────────────────┐
│  Document: {doc_words:,} words = {doc_tokens:,} tokens
│  Model context: {m['tokens']:,} tokens
│  Fits in context? {'✅ YES — use context stuffing' if fits else '❌ NO — you need RAG'}
│
│  Cost per run:  ${cost_per_run:.4f}
│  Monthly ({runs_month} runs): ${cost_month:.2f}
│  vs ChatGPT Pro: $20/month
│  Saving:        ${max(0, 20 - cost_month):.2f}/month
└────────────────────────────────────────┘""")
    if not fits:
        chunks = doc_tokens // (m["tokens"] // 4)
        print(f"  RAG tip: Split into ~{chunks} chunks of {m['tokens']//4:,} tokens each.")
except Exception as e:
    print(f"Using defaults. ({e})")
    doc_tokens = 5000
    cost_month = doc_tokens / 1000 * 0.00015 * 100
    print(f"  Estimated monthly cost (defaults): ${cost_month:.2f}")

<a id='4'></a>
---
## ⚡ Quiz 1 — Term Matching Sprint

**Rules:** Match each term to its correct definition. You have unlimited attempts but try to do it in one go!  
**Scoring:** 1 point per correct match, 8 points total.

Run the cell below and follow the prompts.

In [ ]:
# ─────────────────────────────────────────────
# QUIZ 1 — Term Matching (8 points)
# ─────────────────────────────────────────────
import random

terms_defs = [
    ("Transformer",     "The 2017 neural network architecture 'Attention Is All You Need' — foundation of all modern LLMs"),
    ("Attention",       "Mechanism that weighs relevance of every token to every other token — simultaneously, not sequentially"),
    ("Pre-training",    "Train on billions of internet tokens — done once by the lab, costs millions, you benefit via the API"),
    ("RLHF",            "Phase 2 post-training using human preference data to make models helpful, harmless, and honest"),
    ("GPU",             "Parallel processor chip repurposed for matrix math — a 70B model needs ~140 GB of this"),
    ("Prompt",          "System instructions + examples + context + user question — all sent together in one API call"),
    ("Context window",  "The model's working memory — all tokens it can see at once; Claude 3.5 has 200K tokens"),
    ("RAG",             "Retrieve relevant document chunks before sending to LLM — avoids context overflow on large documents"),
]

random.seed(42)
shuffled = terms_defs.copy()
random.shuffle(shuffled)

defs_only = [d for _, d in shuffled]
terms_only = [t for t, _ in terms_defs]

print("⚡ QUIZ 1 — Term Matching Sprint")
print("=" * 60)
print("\nDefinitions (lettered A–H):")
for i, d in enumerate(defs_only):
    letter = chr(65 + i)
    print(f"  {letter}. {d}")

print("\nTerms to match (enter the letter for each):")
print("  Terms:", ", ".join(terms_only))
print("─" * 60)

score = 0
correct_map = {t: chr(65 + [d for _,d in shuffled].index(d)) for t, d in terms_defs}

for term, correct_def in terms_defs:
    correct_letter = correct_map[term]
    ans = input(f"\nMatch '{term}' → (enter A–H): ").strip().upper()
    if ans == correct_letter:
        print(f"  ✅ Correct!")
        score += 1
    else:
        print(f"  ❌ Wrong. Correct answer: {correct_letter}")
        print(f"     '{term}' = {correct_def[:80]}...")

SCORES["q1_match"] = score
pct = score / 8 * 100
print(f"\n{'='*60}")
print(f"📊 Quiz 1 Score: {score}/8   {color_bar(pct, 30)}")
print(f"   {grade_label(pct)}")
if score < 6:
    print("   💡 Tip: Re-read Section 2 and try again — these terms appear throughout the course.")

⚡ QUIZ 1 — Term Matching Sprint

Definitions (lettered A–H):
  A. Phase 2 post-training using human preference data to make models helpful, harmless, and honest
  B. Parallel processor chip repurposed for matrix math — a 70B model needs ~140 GB of this
  C. The model's working memory — all tokens it can see at once; Claude 3.5 has 200K tokens
  D. Retrieve relevant document chunks before sending to LLM — avoids context overflow on large documents
  E. Train on billions of internet tokens — done once by the lab, costs millions, you benefit via the API
  F. System instructions + examples + context + user question — all sent together in one API call
  G. The 2017 neural network architecture 'Attention Is All You Need' — foundation of all modern LLMs
  H. Mechanism that weighs relevance of every token to every other token — simultaneously, not sequentially

Terms to match (enter the letter for each):
  Terms: Transformer, Attention, Pre-training, RLHF, GPU, Prompt, Context window, RAG
────

<a id='5'></a>
---
## 🆚 5. Context Stuffing vs RAG — The Critical Architectural Decision

This is one of the most important design decisions you'll make as an agent builder.

---

### Approach 1: Context Stuffing (what you build today)

```
 Document text
      │
      ▼
 Paste into prompt ──────────────────────────────────────────┐
      │                                                       │
      ▼                                              ⚠️ If doc > context window:
 LLM reads ALL tokens in one forward pass            ERROR or truncation
      │
      ▼
 Answer sent to email
```

**✅ Pros:** Simple, fast to build, works great for documents under ~80K words  
**❌ Cons:** Breaks when documents exceed context window; all tokens cost money even if irrelevant

---

### Approach 2: RAG — Retrieval-Augmented Generation (next week)

```
 Document
    │
    ▼
 Chunk into ~500-token pieces
    │
    ▼
 Embed each chunk (convert to vector)
    │
    ▼
 Store in vector database (Pinecone, Weaviate, ChromaDB)
    
                    User query
                        │
                        ▼
                 Embed the query
                        │
                        ▼
              Find top-3 similar chunks
                        │
                        ▼
             LLM only sees those 3 chunks  ← much cheaper & scalable!
                        │
                        ▼
                  Answer + citation
```

**✅ Pros:** Scales to millions of documents, cheaper per query, returns citations  
**❌ Cons:** More complex to build; retrieval can miss context that spans chunks

---

### Decision guide

| Situation | Use |
|---|---|
| Single document, < 80K words | **Context stuffing** (today) |
| Document > context window | **RAG** |
| Thousands of documents | **RAG** |
| Need to cite exact source passages | **RAG** |
| Need to stay current without retraining | **RAG** |
| Quick prototype, single PDF | **Context stuffing** |

> 🤔 **Think about it:** What happens if your document is 500,000 words and your context window is 128,000 tokens (~96,000 words)?  
> The document won't fit. You'd get an error or the start of the document gets cut off. **This is exactly why RAG exists.**

<a id='6'></a>
---
## ✏️ Quiz 2 — Context Window & RAG (4 points)

Run the cell below and enter your answers.

In [ ]:
# ─────────────────────────────────────────────
# QUIZ 2 — Context Window & RAG (4 points)
# ─────────────────────────────────────────────

q2 = [
    {
        "q": "Q1. Claude 3.5 Sonnet has a 200K token context window. Roughly how many English words is that?",
        "opts": ["A) ~5,000 words", "B) ~20,000 words", "C) ~150,000 words", "D) ~1,000,000 words"],
        "ans": "C",
        "exp": "200K tokens × 0.75 words/token ≈ 150,000 words. Rule of thumb: 1 token ≈ ¾ of a word."
    },
    {
        "q": "Q2. You have a 10MB PDF (~500,000 words). You want GPT-4o (128K context) to answer questions about it. What should you use?",
        "opts": ["A) Context stuffing — paste the whole thing", "B) RAG — chunk and retrieve relevant sections", "C) Fine-tune the model on the PDF", "D) Summarise the PDF first, then ask questions"],
        "ans": "B",
        "exp": "500K words ≈ 667K tokens — over 5× the context window. RAG retrieves only the relevant chunks, so the LLM only ever sees a manageable amount."
    },
    {
        "q": "Q3. In context stuffing, what happens to the document text after the model generates an answer?",
        "opts": ["A) Stored in the model's long-term memory", "B) Indexed in a vector store automatically", "C) Nothing — processed in one pass, then forgotten", "D) Cached and reused for 24 hours"],
        "ans": "C",
        "exp": "LLMs are stateless. Each API call is a fresh forward pass — no information persists to the next call unless you explicitly pass it in."
    },
    {
        "q": "Q4. In RAG, what does 'embedding' a chunk of text produce?",
        "opts": ["A) A compressed version to save storage", "B) A dense vector of numbers encoding the text's meaning", "C) An encrypted version for security", "D) A direct upload into model weights"],
        "ans": "B",
        "exp": "An embedding is a high-dimensional numeric vector. Similar texts have similar vectors (close in vector space), enabling semantic similarity search."
    },
]

print("✏️  QUIZ 2 — Context Window & RAG")
print("=" * 60)
score = 0
for i, item in enumerate(q2):
    print(f"\n{item['q']}")
    for opt in item['opts']:
        print(f"   {opt}")
    ans = input("   Your answer (A/B/C/D): ").strip().upper()
    if ans == item['ans']:
        print(f"   ✅ Correct! {item['exp']}")
        score += 1
    else:
        print(f"   ❌ Incorrect. Correct answer: {item['ans']}")
        print(f"   💡 {item['exp']}")

SCORES["q2"] = score
pct = score / 4 * 100
print(f"\n{'='*60}")
print(f"📊 Quiz 2 Score: {score}/4   {color_bar(pct, 30)}")
print(f"   {grade_label(pct)}")

<a id='7'></a>
---
## 🔧 7. Building Your First n8n Agent — Step-by-Step

Your document-analyst agent has 5 nodes. Here's exactly what to do:

---

### The workflow

```
┌───────────────┐    ┌───────────────┐    ┌─────────────────────┐    ┌──────────────┐    ┌──────────┐
│  1. Manual    │───▶│  2. Set node  │───▶│  3. OpenAI Chat     │───▶│  4. Gmail    │───▶│  Done!   │
│  Trigger      │    │  Paste doc    │    │  Model (THINK)      │    │  Send answer │    │  ✉️       │
│  (OBSERVE)    │    │  text here    │    │                     │    │  (ACT)       │    │          │
└───────────────┘    └───────────────┘    └─────────────────────┘    └──────────────┘    └──────────┘
```

---

### Step-by-step instructions

**Step 1 — Sign up for n8n**  
Go to [n8n.io](https://n8n.io) → Start for free → Create cloud account (free tier works fine)

**Step 2 — Get your API key**  
- **OpenAI:** [platform.openai.com](https://platform.openai.com) → API Keys → Create new  
- **Gemini (free):** [aistudio.google.com](https://aistudio.google.com) → Get API key (no credit card needed)

**Step 3 — Add credentials in n8n**  
Settings → Credentials → Add → OpenAI (or Google Gemini) → paste your key → Test → Save

**Step 4 — Build the workflow**
1. `+` → Search "Manual Trigger" → Add
2. `+` → Search "Set" → Add → Add field: `document` (string) → Paste your PDF text
3. `+` → Search "OpenAI" → Add → Resource: Message → Model: gpt-4o-mini
4. Set **System prompt:** `You are a document analyst. Answer questions about the document provided. Answer only Yes, No, or a number where possible.`
5. Set **User message:** `Here is the document: {{ $json.document }}. Question: What is the main finding?`
6. `+` → Search "Gmail" → Add → Connect OAuth → Set To: your email
7. Set **Subject:** `Agent Answer` → **Body:** `{{ $json.message.content }}`

**Step 5 — Run it!**  
Click `Test workflow` → Check your inbox → 🎉

---

### Beginner / Intermediate / Advanced tracks

| Level | Task |
|---|---|
| **Beginner** | Follow steps above exactly. Get the email to arrive. |
| **Intermediate** | Modify the system prompt. Add a second question node. Switch to Gemini. |
| **Advanced** | Replace Manual Trigger with Webhook. Parse response as JSON. Add conditional branching. |

In [ ]:
# ─────────────────────────────────────────────
# CELL 7 — Design your system prompt
# ─────────────────────────────────────────────
print("🧰 Prompt Builder for your document-analyst agent")
print("=" * 55)
print("Tip: A well-written system prompt is the difference between")
print("a useful agent and a confusing one.")

role    = input("\nWhat role should the LLM play?\n  (e.g. 'document analyst', 'financial summariser', 'legal assistant')\n  > ") or "document analyst"
domain  = input("\nWhat domain is the document from?\n  (e.g. 'financial reports', 'medical literature', 'legal contracts')\n  > ") or "general documents"
format_ = input("\nHow should it format answers?\n  (e.g. 'Yes/No/number', 'bullet points', 'one sentence')\n  > ") or "Yes, No, or a number where possible"
extra   = input("\nAny extra constraints?\n  (e.g. 'only use information from the document, never guess')\n  > ") or "Only use information from the provided document. Never guess."

system_prompt = f"""You are a {role} specialising in {domain}.
Answer questions about the document provided.
Format: {format_}.
{extra}"""

user_message = """Here is the document:
{{ $json.document }}

Question: [YOUR QUESTION HERE]"""

print("\n" + "=" * 55)
print("✅ Your generated prompts — copy these into n8n:")
print("\n--- SYSTEM PROMPT ---")
print(system_prompt)
print("\n--- USER MESSAGE ---")
print(user_message)
print("\n" + "=" * 55)
print("💡 Paste the system prompt into the 'System' field in the OpenAI node.")
print("   Replace [YOUR QUESTION HERE] with your actual question.")

<a id='8'></a>
---
## ✏️ Quiz 3 — Agent Architecture (4 points)

Run the cell below.

In [ ]:
# ─────────────────────────────────────────────
# QUIZ 3 — Agent Architecture (4 points)
# ─────────────────────────────────────────────

q3 = [
    {
        "q": "Q1. In the n8n document-analyst agent, which node performs the 'Think' step?",
        "opts": ["A) Manual Trigger node", "B) Set node (where you paste the document)", "C) OpenAI Chat Model node", "D) Gmail node"],
        "ans": "C",
        "exp": "The OpenAI (or Gemini) node is the LLM — that's where reasoning happens. The trigger is Observe; Gmail is Act."
    },
    {
        "q": "Q2. Why does the instructor recommend constraining answers to 'Yes, No, or a number'?",
        "opts": ["A) The model can only output short text", "B) Reduces hallucination risk and makes outputs machine-readable", "C) Gmail can only receive short emails", "D) To save API token costs"],
        "ans": "B",
        "exp": "Constrained output formats make it harder to hallucinate a plausible but wrong sentence, and easier to parse or route the answer in the next workflow step."
    },
    {
        "q": "Q3. What is the difference between a system prompt and a user message?",
        "opts": ["A) No difference — they are the same", "B) System prompt sets role/rules; user message is the specific input", "C) User message is sent first; system prompt after", "D) System prompts are only used during fine-tuning"],
        "ans": "B",
        "exp": "System prompt = standing instructions (persona, format rules, constraints). User message = the specific question or task for this run. Both are sent in the same API call."
    },
    {
        "q": "Q4. You want your agent to fire automatically when another app sends an HTTP POST. Which n8n node replaces the Manual Trigger?",
        "opts": ["A) Schedule trigger (Cron)", "B) Webhook trigger", "C) HTTP Request node", "D) Email trigger"],
        "ans": "B",
        "exp": "A Webhook node gives your workflow a unique URL. Any HTTP POST to that URL fires the workflow — perfect for integrating with Postman, a web app, or another automation tool."
    },
]

print("✏️  QUIZ 3 — Agent Architecture")
print("=" * 60)
score = 0
for item in q3:
    print(f"\n{item['q']}")
    for opt in item['opts']:
        print(f"   {opt}")
    ans = input("   Your answer (A/B/C/D): ").strip().upper()
    if ans == item['ans']:
        print(f"   ✅ Correct! {item['exp']}")
        score += 1
    else:
        print(f"   ❌ Incorrect. Correct answer: {item['ans']}")
        print(f"   💡 {item['exp']}")

SCORES["q3"] = score
pct = score / 4 * 100
print(f"\n{'='*60}")
print(f"📊 Quiz 3 Score: {score}/4   {color_bar(pct, 30)}")
print(f"   {grade_label(pct)}")

<a id='9'></a>
---
## 💡 9. Why Build Your Own Agent Instead of Just Using ChatGPT?

The **2-sentence answer** (memorise this):

> *"ChatGPT is a closed product. You cannot connect it to your company's internal database, trigger it from your CRM, or combine it with your existing software — agents built on the API give you that control."*

---

### The builder's advantage in detail

| Dimension | Desktop AI | API-powered agent |
|---|---|---|
| **Integration** | None — copy/paste only | Any system with an API or webhook |
| **Automation** | Manual — you type every time | Event-driven — fires on schedule or trigger |
| **Data privacy** | Your data goes to OpenAI's servers | Can run fully on-premise with open-source models |
| **Cost at scale** | $20/month flat | $0.001–$0.01/run, proportional |
| **Customisation** | Prompt only | Model, prompt, tools, memory, routing — all yours |
| **IP ownership** | Anthropic/OpenAI's infra | Your code, your IP |

---

### The 42-session roadmap — where you're going

```
 Week 1–2  ──▶  Week 3–5  ──▶  Week 6–10  ──▶  Week 11–20  ──▶  Week 21+
   HERE           RAG &          Tool use,        Domain-          Multi-agent
   NOW ⭐         memory         APIs, web         specific         systems &
   Foundation     Multi-step     browsing          agents           evaluation
                  agents         agents            (your use        deployment
                                                   case)
```

> 🏆 **Every agent you will ever build is a variation of what you built today.**  
> More tools, more loops, more memory — but the core Observe → Think → Act pattern never changes.

<a id='10'></a>
---
## 🎓 Quiz 4 — Harder MCQs: Apply What You Know (5 points)

These questions require you to **apply** the concepts, not just recall definitions. Think carefully!

Run the cell below.

In [ ]:
# ─────────────────────────────────────────────
# QUIZ 4 — Applied knowledge (5 points)
# ─────────────────────────────────────────────

q4 = [
    {
        "q": """Q1. A colleague says: 'I'll fine-tune GPT-4 on our 10,000 customer support
   tickets so it knows our product.' What's the key problem for an agent
   that needs up-to-date answers?""",
        "opts": ["A) Fine-tuning is computationally impossible even once",
                 "B) Fine-tuning bakes in a static snapshot — the model can't update when tickets change without retraining",
                 "C) OpenAI doesn't allow fine-tuning on customer data",
                 "D) The model will forget its pre-training if fine-tuned"],
        "ans": "B",
        "exp": "Fine-tuned knowledge is frozen at training time. RAG is far better for dynamic, frequently-updated data — new tickets can be indexed without any model retraining."
    },
    {
        "q": """Q2. Temperature = 0 means the model always picks the highest-probability
   next token. For your document analyst agent, should temperature be
   high or low, and why?""",
        "opts": ["A) High — more creativity gives better answers",
                 "B) Low (near 0) — factual extraction needs determinism, not creativity",
                 "C) Medium (0.5) — always use the middle ground",
                 "D) Temperature doesn't affect document analysis"],
        "ans": "B",
        "exp": "Document fact-extraction is deterministic by nature — you want the most probable (correct) answer, not creative variation. Use temperature 0–0.2 for analytical tasks."
    },
    {
        "q": """Q3. Which statement best describes what RLHF does that pre-training alone
   does NOT?""",
        "opts": ["A) Teaches the model more facts about the world",
                 "B) Increases the model's parameter count",
                 "C) Shapes the model's behaviour to align with human preferences — helpfulness, safety, instruction-following",
                 "D) Gives the model real-time internet access"],
        "ans": "C",
        "exp": "Pre-training teaches the model to predict text. RLHF shapes HOW it responds — making it helpful, honest, and safe. Without RLHF, a model just predicts the next word; it doesn't reliably answer questions or follow instructions."
    },
    {
        "q": """Q4. Your agent costs $0.01/run using GPT-4o. You run it 1,000×/month.
   ChatGPT Pro costs $20/month. What's the monthly API cost, and is it
   cheaper than the subscription?""",
        "opts": ["A) $10 — cheaper than ChatGPT Pro",
                 "B) $100 — more expensive than ChatGPT Pro",
                 "C) $1 — much cheaper, and it runs automatically without you",
                 "D) $1,000 — API is always more expensive"],
        "ans": "A",
        "exp": "1,000 × $0.01 = $10/month. Half the cost of ChatGPT Pro — and the agent runs 24/7 automatically, without you having to type anything."
    },
    {
        "q": """Q5. In a transformer, what does 'multi-head' in multi-head attention mean?""",
        "opts": ["A) The model runs attention multiple times in parallel with different learned weights, capturing different relationship types",
                 "B) The model has multiple output nodes, one per sentence",
                 "C) Attention is applied to multiple documents simultaneously",
                 "D) The model is split across multiple GPUs"],
        "ans": "A",
        "exp": "Multiple attention heads run in parallel, each with different learned projection matrices. One head might focus on syntax, another on coreference, another on semantic roles. Their outputs are concatenated and projected. This is why transformers capture rich, multifaceted relationships."
    },
]

print("🎓 QUIZ 4 — Apply What You Know")
print("=" * 60)
score = 0
for item in q4:
    print(f"\n{item['q']}")
    for opt in item['opts']:
        print(f"   {opt}")
    ans = input("   Your answer (A/B/C/D): ").strip().upper()
    if ans == item['ans']:
        print(f"   ✅ Correct! {item['exp']}")
        score += 1
    else:
        print(f"   ❌ Incorrect. Correct answer: {item['ans']}")
        print(f"   💡 {item['exp']}")

SCORES["q4"] = score
pct = score / 5 * 100
print(f"\n{'='*60}")
print(f"📊 Quiz 4 Score: {score}/5   {color_bar(pct, 30)}")
print(f"   {grade_label(pct)}")

<a id='11'></a>
---
## 📝 11. Homework & Extension Challenges

### Mandatory — complete before next session

- [ ] **1.** Find a document from your domain (< 128K tokens). PDF, Word doc, or plain text.
- [ ] **2.** Upload/paste it into your n8n agent. Write 3 specific questions (Yes/No or numeric preferred).
- [ ] **3.** Run the agent. Screenshot the email you receive with the answers.
- [ ] **4.** Note: were the answers correct? If not — write one sentence on why the LLM might have gotten it wrong.
- [ ] **5.** Post your 3 questions + email screenshot in the class Slack channel.

---

### Optional extensions (celebrated next session!)

- [ ] **A.** Replace OpenAI with Gemini. Compare answers for the same document — are they different?
- [ ] **B.** Add a Webhook trigger. Call your agent with a POST request from Postman.
- [ ] **C.** Handle an edge case: what happens if the document is empty? Add error-handling logic.
- [ ] **D.** Add a second question node — ask 2 different questions and send both answers in one email.

---

### Questions to reflect on (write answers in the cell below)

1. What did the LLM *actually do* with your document? Did it read it? Memorise it?
2. What would break if your document were 200,000 words?
3. How would you modify this workflow to answer your Think Day use case?

In [ ]:
# ─────────────────────────────────────────────
# CELL 11 — Reflection journal (type here!)
# ─────────────────────────────────────────────
print("📓 Reflection Journal — Answer these after running your agent")
print("=" * 60)

r1 = input("\n1. What did the LLM actually do with your document?\n   Did it read it? Memorise it? (Your answer): ")
r2 = input("\n2. What would break if your document were 200,000 words?\n   (Your answer): ")
r3 = input("\n3. How would you modify this workflow for your Think Day use case?\n   (Your answer): ")

print("\n" + "=" * 60)
print("✅ Your reflections (save these for next session):")
print(f"\n1. {r1}")
print(f"\n2. {r2}")
print(f"\n3. {r3}")
print("\n" + "─" * 60)
print("📌 Model answers (read after writing yours):")
print("\n1. The LLM processed ALL tokens in a single forward pass and")
print("   output the answer. Nothing was stored or memorised — the")
print("   next API call starts completely fresh.")
print("\n2. The document (200K words ≈ 267K tokens) would overflow")
print("   most context windows. You'd get an error or truncation.")
print("   Solution: RAG — chunk and retrieve only the relevant bits.")
print("\n3. (Individual — your answer will vary based on your domain!)")

<a id='12'></a>
---
## 🏆 12. Final Score Dashboard

Run the cell below to see your total score across all quizzes.

In [ ]:
# ─────────────────────────────────────────────
# FINAL SCORE DASHBOARD
# ─────────────────────────────────────────────
total = sum(SCORES.values())
max_total = sum(SCORE_MAX.values())
pct = total / max_total * 100

print("=" * 60)
print("🏆  FINAL SCORE DASHBOARD — Agentic AI Day 1 & 2")
print("=" * 60)
print()
print(f"  Quiz 1 — Term Matching:       {SCORES['q1_match']:>2} / {SCORE_MAX['q1_match']}  {color_bar(SCORES['q1_match']/SCORE_MAX['q1_match']*100, 20)}")
print(f"  Quiz 2 — Context & RAG:       {SCORES['q2']:>2} / {SCORE_MAX['q2']}  {color_bar(SCORES['q2']/SCORE_MAX['q2']*100, 20)}")
print(f"  Quiz 3 — Agent Architecture:  {SCORES['q3']:>2} / {SCORE_MAX['q3']}  {color_bar(SCORES['q3']/SCORE_MAX['q3']*100, 20)}")
print(f"  Quiz 4 — Applied Knowledge:   {SCORES['q4']:>2} / {SCORE_MAX['q4']}  {color_bar(SCORES['q4']/SCORE_MAX['q4']*100, 20)}")
print()
print("─" * 60)
print(f"  TOTAL:                        {total:>2} / {max_total}  {color_bar(pct, 20)}")
print(f"  Grade: {grade_label(pct)}")
print()

if pct >= 85:
    print("  🌟 Outstanding! You're ready for RAG and memory next week.")
    print("     Try the Webhook extension challenge before next session.")
elif pct >= 70:
    print("  ✅ Solid foundation! Review any missed questions above.")
    print("     Focus on the RAG diagram in Section 5 before next session.")
elif pct >= 50:
    print("  📈 Good progress. Re-read Sections 2 and 5, then re-run")
    print("     the quizzes — you can run each cell as many times as you like.")
else:
    print("  📚 Keep going! Read each theory section carefully, then")
    print("     re-run the quiz cells. All explanations are shown after each answer.")

print()
print("=" * 60)
print("📌 Two-day achievement summary:")
print("  You went from LLM theory to a live, email-sending agent.")
print("  Every agent you build from here is a variation of that core loop.")
print("=" * 60)